## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [1]:
# loading text/data from a file(pdf)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader("../data/", glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = dir_loader.load()
print(documents)

/tmp/ipykernel_22337/976910581.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': '2026-11-01T00:41:57+05:30', 'source': '../data/Electric_Motors.pdf', 'total_pages': 229, 'page': 0, 'page_label': '1'}, page_content='1\nELECTRIC MOTORS\nINTRODUCTION\nElectric motors are so much a part of everyday life that we seldom\ngive them a second thought. When we switch on an electric drill,\nfor example, we con Wdently expect it to run rapidly up to the correct\nspeed, and we do not question how it knows what speed to run at, or\nhow it is that once enough energy has been drawn from the supply to\nbring it up to speed, the power drawn falls to a very low level. When we\nput the drill to work it draws more power, and when we Wnish the power\ndrawn from the mains reduces automatically, without intervention on\nour part.\nThe humble motor, consisting of nothing more than an arrangement\nof copper coils and steel laminations, is clearly rather a clever energy\nconverter, which warrants serious consider

In [2]:
# creating chunks from text loaded

from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_and_chunk(documents, chunk_size=1000, chunk_overlap=500):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# chunks

split 229 documents into 851 chunks.


In [3]:
# embedding
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, PreTrainedModel
import numpy as np
from typing import List, Any, Dict


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"model loaded with embedding dimentions: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} chunks..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

/home/ayush/Desktop/projects/rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading embedding manager: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2414.90it/s]


model loaded with embedding dimentions: 384


In [4]:
# vector store
import os
import chromadb
import hashlib
import numpy as np
from pathlib import Path

class VectorManager:
    def __init__(self, collection_name=None, persist_directory="../data/vector_store"):
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        source_files = sorted({Path(doc.metadata["source"]).resolve() for doc in documents})
        digest = hashlib.sha256(embedding_manager.model_name.encode("utf-8"))
        for source_file in source_files:
            digest.update(str(source_file).encode("utf-8"))
            digest.update(source_file.read_bytes())
        self.collection_name = collection_name or f"motor_docs_{digest.hexdigest()[:12]}"
        self._load_store()

    def _load_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "electric motor reference PDFs", "hnsw:space": "cosine"},
            )
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error loading vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("no of documents does not match no of embedding")
        ids, metadatas, documents_text, embeddings_list = [], [], [], []
        for doc, embedding in zip(documents, embeddings):
            source = str(doc.metadata.get("source", "unknown"))
            page = doc.metadata.get("page", 0)
            doc_hash = hashlib.sha256(doc.page_content.encode("utf-8")).hexdigest()[:16]
            doc_id = hashlib.sha256(f"{source}:{page}:{doc_hash}".encode("utf-8")).hexdigest()[:24]
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        self.collection.upsert(ids=ids, metadatas=metadatas, documents=documents_text, embeddings=embeddings_list)
        print(f"added {len(documents)} documents into vector store")
        print(f"total documents in collection: {self.collection.count()}")

vector_store=VectorManager()
vector_store


vector store initialized, collection: motor_docs_c6e2b23be565
existing documents in collection: 851


In [5]:
# convert text into embeddings
texts = [doc.page_content for doc in chunks]

# generate embedding
embeddings = embedding_manager.generate_embedding(texts)

# store in vector db
vector_store.add_documents(chunks, embeddings)

generating embedding for 851 chunks..


Batches: 100%|██████████| 27/27 [01:50<00:00,  4.10s/it]


generated embeddings with shape: (851, 384)
added 851 documents into vector store
total documents in collection: 851


### retrieval pipeline

In [16]:
class RagRetriever:
    def __init__(self, vector_store: VectorManager, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.30) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
        retrieved_docs = []
        documents = results["documents"][0] or []
        metadatas = results["metadatas"][0] or []
        distances = results["distances"][0] or []
        ids = results["ids"][0] or []
        for rank, (doc_id, document, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances), start=1
        ):
            similarity_score = 1 - float(distance)
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": doc_id, "content": document, "metadata": metadata,
                    "similarity_score": similarity_score, "rank": rank,
                })
        return retrieved_docs

rag_retriever = RagRetriever(vector_store, embedding_manager)
rag_retriever


### vector store to llm output generation

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API"))

True

In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a strict QA assistant. Answer the user's question using only the provided context. If the answer cannot be found in the context, reply exactly with: 'I do not have access to this information.' Do not use any outside knowledge.

Context:
{context}

Question: {question}
"""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
        

In [19]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [10]:
rag_retriever.retrieve("electric motors")

generating embedding for 1 chunks..


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.56it/s]

generated embeddings with shape: (1, 384)


[{'id': '57d5ebfe4b4bf08d7754b2cd',
  'content': 'preferred, despite its larger size.\nENERGY CONVERSION – MOTIONAL EMF\nWe now turn away from considerations of what determines the overall\ncapability of a motor to what is almost the other extreme, by examining\nthe behaviour of a primitive linear machine which, despite its obvious\nsimplicity, encapsulates all the key electromagnetic energy conversion\nprocesses that take place in electric motors. We will see how the process\nof conversion of energy from electrical to mechanical form is elegantly\nrepresented in an ‘equivalent circuit’ from which all the key aspects of\nmotor behaviour can be predicted. This circuit will provide answers to\nsuch questions as ‘how does the motor automatically draw in more\npower when it is required to work’, and ‘what determines the steady\nElectric Motors 25',
  'metadata': {'content_length': 792,
   'source': '../data/Electric_Motors.pdf',
   'total_pages': 229,
   'creationdate': '2026-11-01T00:41:5

In [20]:
# llm integration
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
import re
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API") or os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)
NO_ANSWER = "I couldn't find that in the electric motor reference material."

def format_bare_math(text):
    text = re.sub(
        r"\\\[(.*?)\\\]",
        lambda match: "$$\n" + match.group(1).strip() + "\n$$",
        text,
        flags=re.DOTALL,
    )
    text = re.sub(
        r"\\\((.*?)\\\)",
        lambda match: "$" + match.group(1).strip() + "$",
        text,
        flags=re.DOTALL,
    )

    def wrap_bare_formula(match):
        formula = match.group(1).strip().rstrip(" ;,.")
        return f"${formula}$"

    return re.sub(
        r"(?<!\\)\[\s*([^\[\]\n]*?(?:\\[A-Za-z]+|=)[^\[\]\n]*?)\s*(?<!\\)\]",
        wrap_bare_formula,
        text,
    )

def rag_simple(query, retriever, llm, top_k=5, min_score=0.30):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return NO_ANSWER

    context = "\n\n".join(
        f"[{index}] {doc['content']}" for index, doc in enumerate(results, start=1)
    )
    prompt = f"""You answer questions about electric motors from the reference passages below.
Use only facts supported by these passages. Match the meaning of the question even when it is phrased simply or uses different words than a passage.
Answer directly when a passage contains the answer. Do not abstain merely because the question is short.
If no passage contains the answer, reply exactly: {NO_ANSWER}
Include the supporting passage number, such as [1].
Write equations in LaTeX using $...$ inline or $$...$$ on separate lines; do not use bare square brackets as math delimiters.

Passages:
{context}

Question: {query}
Answer:"""
    response = llm.invoke([prompt])
    answer = response.content
    if not isinstance(answer, str):
        answer = "".join(str(part) for part in answer)
    answer = answer.strip()
    if answer == NO_ANSWER and results[0]["similarity_score"] >= 0.50:
        answer = f"{results[0]['content'].strip()} [1]"
    return format_bare_math(answer)


In [32]:
answer = rag_simple("steady-state operating characteristics of series motor", rag_retriever, llm)
from IPython.display import Markdown, display
display(Markdown(answer))

generating embedding for 1 chunks..


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.49it/s]

generated embeddings with shape: (1, 384)


Series motors exhibit a steep speed‑load characteristic: as load torque increases, the speed falls rapidly. This makes them unsuitable for constant‑speed applications and is why they are typically used for traction, where variable speed with load is acceptable. [5]